# LSTM T2D -- Multi-Horizon Evaluation (5y / 3y / 2y / 1y)

Corre esto **después** de `LSTM_T2D_Training.ipynb` -- usa el modelo ya entrenado (`models/lstm_model_regular_1year_best.pth`), no vuelve a entrenar.

**Qué responde**: "este modelo, entrenado con todo el histórico disponible antes del diagnóstico, ¿sigue prediciendo bien si en el momento de evaluar solo le doy información de hasta 5 (o 3, 2, 1) años antes del diagnóstico?" -- **un solo modelo**, evaluado sobre 4 test sets censurados a distinta distancia del diagnóstico. Igual que tu script `evaluate_and_shap.py` del BERT.

**Población de test**: el test set **externo** (`matched_test.csv`) -- es el único conjunto fijo que tenemos (el interno se re-divide en cada una de las 10 iteraciones de entrenamiento, así que no hay "un" test set interno único al que aplicar esto).

**Regla de censura** (misma que el BERT, Opción A):
- Casos (diabéticos): cutoff = fecha de diagnóstico − N años
- Controles: cutoff = fecha de su última visita registrada − N años
- Se descartan los pacientes que se queden con menos de 2 visitas tras aplicar el cutoff (mismo criterio mínimo que usa el BERT)

**Limitación metodológica heredada del BERT** (documéntala igual en el TFM): al no emparejar casos y controles por fecha índice, los controles con historiales más cortos pueden perder más pacientes al aplicar el cutoff que los casos -- la muestra de controles superviviente podría sesgarse hacia pacientes con seguimiento más largo. El tamaño de muestra se irá reduciendo según crece el horizonte (5 años < 1 año) -- es esperado, no un error.


In [ ]:
# %% Mount Google Drive and set the working folder
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_FOLDER = "/content/drive/MyDrive/Colab Notebooks/LSTM"
os.chdir(DRIVE_FOLDER)
print("Working directory:", os.getcwd())


In [ ]:
# %% Install / upgrade packages
!pip install -q --upgrade pandas
!pip install -q shap imbalanced-learn

import pandas as pd
print("pandas version:", pd.__version__)
print("If this is the FIRST time running this cell in this session: Runtime > Restart session, then Runtime > Run all.")


In [ ]:
# %% Imports
import os
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              confusion_matrix, ConfusionMatrixDisplay, roc_auc_score)
from imblearn.under_sampling import RandomUnderSampler
import matplotlib.pyplot as plt
import shap
import tqdm

def load_streamed_patient_dict(path, desc="Loading"):
    with open(path, "rb") as f:
        first_obj = pickle.load(f)
        if isinstance(first_obj, dict):
            return first_obj
        n_patients = first_obj
        patient_data = {}
        for _ in tqdm.tqdm(range(n_patients), desc=desc, colour='blue'):
            pid, df = pickle.load(f)
            patient_data[pid] = df
        return patient_data

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Using device: {DEVICE}")


In [ ]:
# %% Config
DATA_PATH = "regular_patients_1_year.pkl"
EARLIEST_DX_PATH = "EARLIEST_DX_deid.csv"
EXTERNAL_TEST_PATH = "matched_test.csv"
MODEL_PATH = "models/lstm_model_regular_1year_best.pth"
OUTPUT_DIR = "models"

HORIZONS_YEARS = [5, 3, 2, 1]
MIN_VISITS_AT_HORIZON = 2      # same minimum-context filter as the BERT script
RANDOM_STATE = 42

RUN_SHAP_PER_HORIZON = True    # set False to skip SHAP and only get metrics (much faster)
SHAP_N_PATIENTS_PER_HORIZON = 40
SHAP_TOP_N_FEATURES = 25
KERNEL_SHAP_NSAMPLES = "auto"

os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
# %% Load the trained model + its architecture config
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
saved_config = checkpoint['config']
print(f"Loaded checkpoint: iteration {checkpoint['iteration']}, test AUC {checkpoint['test_auc']:.4f}")

class ConfigNS:
    pass
config = ConfigNS()
for k, v in saved_config.items():
    setattr(config, k, v)


class PatientRiskModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=config.SEQ_INPUT_SIZE, hidden_size=config.LSTM_HIDDEN_SIZE,
            num_layers=config.NUM_LAYERS, batch_first=True,
            dropout=config.DROPOUT if config.NUM_LAYERS > 1 else 0.0
        )
        self.demo_encoder = nn.Sequential(
            nn.Linear(config.DEMO_INPUT_SIZE, config.DEMO_HIDDEN_SIZE), nn.ReLU(), nn.Dropout(config.DROPOUT)
        )
        self.classifier = nn.Sequential(
            nn.Linear(config.LSTM_HIDDEN_SIZE + config.DEMO_HIDDEN_SIZE, config.COMBINED_HIDDEN_SIZE),
            nn.ReLU(), nn.Dropout(config.DROPOUT), nn.Linear(config.COMBINED_HIDDEN_SIZE, 1)
        )

    def forward(self, x_seq, x_demo, lengths):
        packed_x = pack_padded_sequence(x_seq, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed_x)
        seq_embedding = h_n[-1]
        demo_embedding = self.demo_encoder(x_demo)
        combined = torch.cat((seq_embedding, demo_embedding), dim=1)
        return self.classifier(combined).squeeze(-1)


model = PatientRiskModel(config).to(DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(" Model loaded and ready.")


In [ ]:
# %% Load patient data, diagnosis dates (deduped -- MINIMUM to avoid the
# leakage bug), and the external test population
imputed_patients_matrices_all = load_streamed_patient_dict(DATA_PATH, desc="Loading patient data")

df2 = pd.read_csv(EARLIEST_DX_PATH)
df2 = df2.drop(columns=['DX'], errors='ignore')
df2['EARLIEST_DX'] = pd.to_datetime(df2['EARLIEST_DX'], errors='coerce')
df2_earliest = df2.sort_values('EARLIEST_DX').drop_duplicates(subset='PATIENT_ID', keep='first')
assert df2_earliest['PATIENT_ID'].is_unique, \
    "Duplicate PATIENT_IDs after dedup -- DO NOT PROCEED, this is the leakage bug."
imputed_patients = set(imputed_patients_matrices_all.keys())
df2_filtered = df2_earliest[df2_earliest['PATIENT_ID'].isin(imputed_patients)]
dx_date_dict = df2_filtered.set_index('PATIENT_ID')['EARLIEST_DX'].dt.date.to_dict()
print(f" {len(dx_date_dict)} patients have a matched (deduped) diagnosis date.")

ext_df = pd.read_csv(EXTERNAL_TEST_PATH)
external_test_ids = set(ext_df['PATIENT_ID'].astype(str).unique())
print(f" External test population: {len(external_test_ids)} patient IDs.")

seq_indices = [idx for idx in imputed_patients_matrices_all[next(iter(imputed_patients_matrices_all))].index
               if idx not in config.DEMO_COLUMNS]
assert len(seq_indices) == config.ORIGINAL_SEQ_SIZE, \
    f"Feature count mismatch: data has {len(seq_indices)}, model expects {config.ORIGINAL_SEQ_SIZE}."
N_SEQ = config.ORIGINAL_SEQ_SIZE
feature_names = [str(idx) for idx in seq_indices] + list(config.DEMO_COLUMNS)


In [ ]:
# %% Reference dates per external-test patient (cases: diagnosis date;
# controls: their own last visit date) -- same rule as the BERT script.
reference_dates = {}
for pid in external_test_ids:
    pid_int = int(pid) if str(pid).isdigit() else pid
    if pid_int not in imputed_patients_matrices_all:
        continue
    df_patient = imputed_patients_matrices_all[pid_int]
    visit_dates = sorted(pd.to_datetime(col) for col in df_patient.columns)
    if not visit_dates:
        continue
    if pid_int in dx_date_dict:
        reference_dates[pid_int] = (pd.Timestamp(dx_date_dict[pid_int]), 1)  # (ref_date, label)
    else:
        reference_dates[pid_int] = (visit_dates[-1], 0)

print(f" Reference dates computed for {len(reference_dates)} external-test patients "
      f"({sum(1 for _, l in reference_dates.values() if l == 1)} cases, "
      f"{sum(1 for _, l in reference_dates.values() if l == 0)} controls).")


In [ ]:
# %% Build a horizon-censored dataset for one N-year cutoff
def build_horizon_dataset(years):
    records = {}  # pid -> (masked_features, demo_features, label)
    for pid, (ref_date, label) in reference_dates.items():
        df_patient = imputed_patients_matrices_all[pid]
        cutoff = ref_date - pd.DateOffset(years=years)

        visit_cols = [(pd.to_datetime(col), col) for col in df_patient.columns]
        surviving_cols = [orig_col for dt, orig_col in visit_cols if dt < cutoff]
        if len(surviving_cols) < MIN_VISITS_AT_HORIZON:
            continue

        df_h = df_patient[surviving_cols]
        demo_features = df_h.loc[config.DEMO_COLUMNS].iloc[:, 0].values
        seq_raw = df_h.loc[seq_indices].T.values
        mask = (~np.isnan(seq_raw)).astype(np.float32)
        seq_vals = np.nan_to_num(seq_raw, nan=0.0)
        masked_features = np.concatenate([seq_vals, mask], axis=1)

        records[pid] = (masked_features, demo_features, label)
    return records


In [ ]:
# %% PyTorch Dataset/DataLoader + evaluate() (same as training notebook, plus specificity)
class PatientDataset(Dataset):
    def __init__(self, seq_data, demo_data, labels):
        self.seq_data, self.demo_data, self.labels = seq_data, demo_data, labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx): return self.seq_data[idx], self.demo_data[idx], self.labels[idx]

def collate_fn(batch):
    seq_batch, demo_batch, labels_batch = zip(*batch)
    seq_tensors = [torch.tensor(x, dtype=torch.float32) for x in seq_batch]
    lengths = torch.tensor([len(x) for x in seq_tensors])
    seq_padded = pad_sequence(seq_tensors, batch_first=True, padding_value=0.0)
    demo_tensor = torch.tensor(np.array(demo_batch), dtype=torch.float32)
    labels_tensor = torch.tensor(labels_batch, dtype=torch.float32)
    return seq_padded, demo_tensor, labels_tensor, lengths


def evaluate(model, data_loader, title="Evaluation"):
    model = model.to(DEVICE)
    model.eval()
    y_true, y_prob = [], []
    with torch.no_grad():
        for seq_batch, demo_batch, y_batch, lengths in data_loader:
            seq_batch, demo_batch = seq_batch.to(DEVICE), demo_batch.to(DEVICE)
            logits = model(seq_batch, demo_batch, lengths)
            y_true.extend(y_batch.tolist())
            y_prob.extend(torch.sigmoid(logits).tolist())

    y_pred = [1 if p > 0.5 else 0 for p in y_prob]
    acc = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob) if len(set(y_true)) > 1 else float('nan')
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    cm = confusion_matrix(y_true, y_pred)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    else:
        specificity = 0.0

    print(f"\n --- {title} ---")
    print(f"n={len(y_true)} | ACC:{acc:.4f} AUC:{auc:.4f} PREC:{prec:.4f} REC:{rec:.4f} SPEC:{specificity:.4f} F1:{f1_macro:.4f}")
    return acc, auc, prec, rec, f1_macro, specificity, cm


In [ ]:
# %% Evaluate the model at each horizon (balanced 1:1, same as everywhere
# else in this pipeline)
horizon_records = {}  # years -> patient_records dict (kept for the SHAP cell below)
metrics_rows = []

for years in HORIZONS_YEARS:
    print(f"\n{'='*20} HORIZON: {years} year(s) {'='*20}")
    records = build_horizon_dataset(years)
    horizon_records[years] = records

    labels = [label for _, _, label in records.values()]
    n_pos, n_neg = sum(1 for l in labels if l == 1), sum(1 for l in labels if l == 0)
    print(f" {len(records)} patients survive this horizon's cutoff ({n_pos} cases / {n_neg} controls).")

    if n_pos == 0 or n_neg == 0:
        print(f" Only one class present at this horizon -- skipping (undefined AUC).")
        continue

    pids_h = list(records.keys())
    seqs_h = [records[p][0] for p in pids_h]
    demos_h = [records[p][1] for p in pids_h]
    labels_h = [records[p][2] for p in pids_h]

    idx_arr = np.arange(len(labels_h)).reshape(-1, 1)
    rus = RandomUnderSampler(sampling_strategy=1.0, random_state=RANDOM_STATE)
    resampled_idx, y_bal = rus.fit_resample(idx_arr, labels_h)
    resampled_idx = resampled_idx.flatten()
    y_bal = y_bal if isinstance(y_bal, list) else y_bal.tolist()
    seqs_bal = [seqs_h[i] for i in resampled_idx]
    demos_bal = [demos_h[i] for i in resampled_idx]

    loader_h = DataLoader(PatientDataset(seqs_bal, demos_bal, y_bal), batch_size=128, collate_fn=collate_fn)
    acc, auc, prec, rec, f1, spec, cm = evaluate(model, loader_h, f"{years}-year horizon (balanced, n={len(y_bal)})")

    metrics_rows.append({
        'horizon': f'{years}y', 'n_patients': len(y_bal),
        'n_cases': sum(1 for y in y_bal if y == 1), 'n_controls': sum(1 for y in y_bal if y == 0),
        'roc_auc': auc, 'f1': f1, 'precision': prec, 'recall': rec, 'specificity': spec,
    })

    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No T2D', 'T2D'])
    fig, ax = plt.subplots(figsize=(5, 5))
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f"Confusion Matrix - {years}y horizon")
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, f"confusion_matrix_{years}y.png"), dpi=200)
    plt.show()


In [ ]:
# %% Save the horizon comparison table (same filename convention as the BERT script)
metrics_df = pd.DataFrame(metrics_rows)
metrics_path = os.path.join(OUTPUT_DIR, "metrics_by_horizon.csv")
metrics_df.to_csv(metrics_path, index=False)
print(f"\nSaved comparison table -> {metrics_path}")
metrics_df


In [ ]:
# %% SHAP per horizon (optional -- set RUN_SHAP_PER_HORIZON=False above to skip)
# Same masking-based mechanism as LSTM_SHAP_Analysis.ipynb: each raw feature
# is one Shapley player, "off" = missing (value=0, mask=0) across the WHOLE
# (horizon-truncated) visit history at once. Output format matches the BERT
# script's per-horizon shap_top_codes_<horizon>.csv/.png.

def make_patient_predict_fn(masked_features, demo_features):
    n_timesteps = masked_features.shape[0]
    lengths = torch.tensor([n_timesteps])
    def f(Z):
        n = Z.shape[0]
        batch_seq = np.zeros((n, n_timesteps, N_SEQ * 2), dtype=np.float32)
        batch_demo = np.zeros((n, config.DEMO_INPUT_SIZE), dtype=np.float32)
        for i in range(n):
            z = Z[i]
            m = masked_features.copy()
            off_features = np.where(z[:N_SEQ] == 0)[0]
            if len(off_features) > 0:
                m[:, off_features] = 0.0
                m[:, N_SEQ + off_features] = 0.0
            batch_seq[i] = m
            batch_demo[i] = demo_features if z[N_SEQ] == 1 else 0.0
        preds = []
        BATCH = 256
        with torch.no_grad():
            for start in range(0, n, BATCH):
                end = min(start + BATCH, n)
                bs = torch.tensor(batch_seq[start:end]).to(DEVICE)
                bd = torch.tensor(batch_demo[start:end]).to(DEVICE)
                bl = lengths.repeat(end - start)
                out = model(bs, bd, bl)
                preds.append(torch.sigmoid(out).cpu().numpy())
        return np.concatenate(preds)
    return f


if RUN_SHAP_PER_HORIZON:
    M = N_SEQ + config.DEMO_INPUT_SIZE
    background = np.zeros((1, M))

    for years in HORIZONS_YEARS:
        records = horizon_records.get(years, {})
        if not records:
            continue
        print(f"\n{'='*20} SHAP -- {years}-year horizon {'='*20}")

        rng = np.random.RandomState(42)
        pos_ids = [p for p, (_, _, y) in records.items() if y == 1]
        neg_ids = [p for p, (_, _, y) in records.items() if y == 0]
        n_each = SHAP_N_PATIENTS_PER_HORIZON // 2
        if len(pos_ids) == 0 or len(neg_ids) == 0:
            print("  Only one class present -- skipping SHAP for this horizon.")
            continue
        sample_pids_h = (list(rng.choice(pos_ids, size=min(n_each, len(pos_ids)), replace=False)) +
                          list(rng.choice(neg_ids, size=min(n_each, len(neg_ids)), replace=False)))

        all_sv = np.zeros((len(sample_pids_h), M))
        for row_i, pid in enumerate(tqdm.tqdm(sample_pids_h, desc=f"SHAP {years}y", colour='green')):
            masked_features, demo_features, label = records[pid]
            f_patient = make_patient_predict_fn(masked_features, demo_features)
            explainer = shap.KernelExplainer(f_patient, background)
            sv = explainer.shap_values(np.ones((1, M)), nsamples=KERNEL_SHAP_NSAMPLES, silent=True)
            all_sv[row_i] = np.array(sv).flatten()

        n_present = np.zeros(M, dtype=int)
        for pid in sample_pids_h:
            masked_features, _, _ = records[pid]
            for j in range(N_SEQ):
                if masked_features[:, N_SEQ + j].max() > 0:
                    n_present[j] += 1
        n_present[N_SEQ:] = len(sample_pids_h)

        imp_df = pd.DataFrame({
            'feature': feature_names, 'mean_shap': all_sv.mean(axis=0),
            'mean_abs_shap': np.abs(all_sv).mean(axis=0), 'n_occurrences': n_present,
        }).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
        imp_df.to_csv(os.path.join(OUTPUT_DIR, f"shap_top_features_{years}y.csv"), index=False)

        top = imp_df.head(SHAP_TOP_N_FEATURES).iloc[::-1]
        colors = ["#d62728" if v > 0 else "#1f77b4" for v in top['mean_shap']]
        fig, ax = plt.subplots(figsize=(8, max(6, SHAP_TOP_N_FEATURES * 0.3)))
        ax.barh(top['feature'], top['mean_shap'], color=colors)
        ax.axvline(0, color='black', linewidth=0.8)
        ax.set_xlabel("Mean SHAP value (-> T2D risk)")
        ax.set_title(f"Top {SHAP_TOP_N_FEATURES} features by |SHAP| -- {years}y horizon (LSTM)")
        fig.tight_layout()
        fig.savefig(os.path.join(OUTPUT_DIR, f"shap_top_features_{years}y.png"), dpi=200)
        plt.show()
        print(f"  Saved shap_top_features_{years}y.csv / .png")
